# Clone Repository and set up the Environment

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

Cloning into 'robust-eeg-models'...
remote: Enumerating objects: 3273, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 3273 (delta 51), reused 37 (delta 37), pack-reused 3214 (from 3)
Receiving objects: 100% (3273/3273), 832.23 MiB | 47.77 MiB/s, done.
Resolving deltas: 100% (1248/1248), done.
Updating files: 100% (2211/2211), done.
Filtering content: 100% (2/2), 663.03 MiB | 100.66 MiB/s, done.


In [3]:
%cd robust-eeg-models

/content/robust-eeg-models


In [5]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [6]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna
!pip install torchattacks
!pip install advertorch
!pip install foolbox
!pip install captum

# Clean uninstall of briandecode
!pip uninstall -y braindecode

# Install latest braindecode and autoattack code from GitHub (which includes braindecode CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir
!pip install git+https://github.com/fra31/auto-attack

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 92.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 159.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 162.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: seaborn
    Found existing installation: seaborn 0.13.2
    Uninstalling seaborn-0.13.2:
      Successfully uninstalled seaborn-0.13.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [moabb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
  Attempting uninstall: chardet
    Found existing installation: chardet 5.2.0
    Uninstalling chardet-5.2.0:
      Successfully uninstalled chardet-5.2.0
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [torchattacks]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moabb 1.2.0 requires requests<3.0.0,>=2.28.1, but you have requests 2.25.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.25.1 which is incompatible.
sphinx 8.2.3 requires requests>=2.30.0, but you h

In [6]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.2.0
CTNet is available ✅


In [7]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # or ":16:8" if memory is tight
os.environ["PYTHONHASHSEED"] = "0"                  # optional, extra stability


In [8]:
import torch, torchattacks, foolbox, optuna, autoattack, captum
# import advertorch
import importlib, sys, pickle, json, random, time, datetime, numbers, hashlib, subprocess

import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [9]:
from datetime import datetime
from collections import defaultdict

from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, CosineAnnealingWarmRestarts

from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity

# Use to visualise the embeddings, COULD be used to visualise the explanations
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

#EEGMamba-MOE approximation
from models.eeg_mamba_fft import create_eegmamba, EEGMamba

# Loading data for training

In [53]:
#import numpy as np
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
from braindecode.preprocessing import create_windows_from_events
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset
from torch.utils.data import Subset

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = MOABBDataset(
    dataset_name='BNCI2014001', subject_ids=[1]
)

#----------------------------------------------------------------------
# After loading we preprocess

low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

#-----------------------------------------------------------------------

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples = int(-0.5 * sfreq) , # -0.5s before cue
    trial_stop_offset_samples = 0,   # 4.0s after cue (t=2s to t=6s)
    preload=True,
    # verbose=0
)


# ----------------------------------------------------------------------
# Split into train and test
splitted = windows_dataset.split("session")
train_set = splitted["0train"]  # Session train
test_set = splitted["1test"]  # Session evaluation

# ----------------------------------------------------------------------
# Split into train, val subsets

X_train = SliceDataset(train_set, idx=0)

y_train = np.array([y for y in SliceDataset(train_set, idx=1)])

X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
y_test = np.array(test_set.get_metadata().target)                   # (N,)

train_indices, val_indices = train_test_split(
      X_train.indices_, test_size=0.2, shuffle=False
  )
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)

# Build simple tensors to compute stats on train windows only
X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T
train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

# Optionally, keep empirical bounds for later clipping in attack
train_min = X_train.min(axis=(0,2), keepdims=True)
train_max = X_train.max(axis=(0,2), keepdims=True)


# 3) materialize test tensors (NOT SliceDataset)
X = torch.tensor(X_test, dtype=torch.float32, device=device)
y = torch.tensor(y_test, dtype=torch.long, device=device)

x, y , meta = train_set[0]
print(type(x), x.shape, x.mean(), x.std())

# print(train_mean.shape)
# print(train_std.shape)
# print(train_min.shape)
# print(train_max.shape)
# print(train_mean)
# print(train_std)
# print(train_min)
# print(train_max)

# Save these via _save_run(... train_mean=train_mean, train_std=train_std, train_min=train_min, train_max=train_max)


/usr/local/lib/python3.12/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
<class 'numpy.ndarray'> (22, 1125) -0.002018633 1.0159434


Check if this loads as train_set = x, y or x, y , meta

---
Useful for later

In [12]:
sample = train_set[0]
print(type(sample))
print(len(sample))   # see what fields exist ()

<class 'tuple'>
3


## Set loading functions

In [55]:
def load_subject_data_cached(dataset, subject_id):
    cache_file = f'cache/subject_{subject_id}_processed.pkl'

    if os.path.exists(cache_file):
        # Load from cache - instant!
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    # Process and cache
    train_set, test_set, train_subset, val_subset = load_subject_data(dataset,subject_id)

    os.makedirs('cache', exist_ok=True)
    with open(cache_file, 'wb') as f:
        pickle.dump((train_set, test_set, train_subset, val_subset), f)


    return train_set, test_set, train_subset, val_subset

def load_subject_data(dataset, subject_id):

    import numpy as np
    from braindecode.datasets import MOABBDataset
    from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
    from braindecode.preprocessing import create_windows_from_events
    from sklearn.model_selection import train_test_split
    from skorch.helper import SliceDataset
    from torch.utils.data import Subset

    dataset = MOABBDataset(
        dataset_name=dataset, subject_ids=[subject_id]
    )

    #----------------------------------------------------------------------
    # After loading we preprocess

    low_cut_hz = 4.0  # low cut frequency for filtering
    high_cut_hz = 38.0  # high cut frequency for filtering
    # Parameters for exponential moving standardization
    factor_new = 1e-3
    init_block_size = 750

    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
        Preprocessor(
            lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
            factor=1e6,
        ),
        Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
        Preprocessor(
            exponential_moving_standardize,  # Exponential moving standardization
            factor_new=factor_new,
            init_block_size=init_block_size,
        ),
    ]

    # Preprocess the data
    preprocess(dataset, preprocessors, n_jobs=-1)

    #-----------------------------------------------------------------------

    trial_start_offset_seconds = -0.5
    # Extract sampling frequency, check that they are same in all datasets
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
    # Calculate the window start offset in samples.
    trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

    # Create windows using braindecode function for this. It needs parameters to
    # define how windows should be used.
    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=0,
        preload=True,

    )

    # ----------------------------------------------------------------------
    # Split into train and test
    splitted = windows_dataset.split("session")
    train_set = splitted["0train"]  # Session train
    test_set = splitted["1test"]  # Session evaluation

    # ----------------------------------------------------------------------
    # Split into train, val subsets

    X_train = SliceDataset(train_set, idx=0)
    y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
    train_indices, val_indices = train_test_split(
        X_train.indices_, test_size=0.2, shuffle=False
    )
    train_subset = Subset(train_set, train_indices)
    val_subset = Subset(train_set, val_indices)

    return train_set, test_set, train_subset, val_subset


## Data loading sanity check

In [66]:
# Run once or sanity check
subject_id = 1
train_set, test_set, train_subset, val_subset = load_subject_data_cached("BNCI2014001", subject_id)
print(f"Window shape: {windows_dataset[0][0].shape}")
train_subset[0][0].shape[0]

Window shape: (22, 1125)


22

## Load and inspect model

In [15]:
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.12/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


# Training

## Set model hyper params

In [ ]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}
alt_params = {'lr': 1e-3, 'batch_size': 128, 'weight_decay': 5e-3, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

## Single run mode

Single training run for model debugging, sanity checks and testing non braindecode architectures (EEGMammba)



In [7]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

device = "cuda" if torch.cuda.is_available() else "cpu"


# Load tehe training data
train_set, test_set, train_subset, val_subset= load_subject_data_cached("BNCI2014001", 1)

# Build transforms list
transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]


# Extract model params from dataset, initialise model and set hyper-parameters
classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
n_classes = len(classes)
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,


)

# Hyper params
params = mamba_params

# Toggle this to enable mamba (For EEGMamba only)
# model.enable_moe(True)

# Enable MoE head instead of standard classifier (For EEGMamba only)
# model.use_moe = False

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def make_scheduler(optimizer, last_epoch=-1):
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=10, last_epoch=last_epoch)
    cosine = CosineAnnealingLR(optimizer, T_max=n_epochs-10, last_epoch=last_epoch)
    return CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    # iterator_train=AugmentedDataLoader,
    # iterator_train__transforms=transforms,
    # iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],

    # Looking at subjects
    # optimizer = torch.optim.SGD,
    # optimizer__momentum=0.9,

    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=[
        "accuracy",
        # ("lr_scheduler", LRScheduler(make_scheduler))
        # ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=200, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=500,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

import numbers


NameError: name 'torch' is not defined

# Record Baselines for chosen models

In [11]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)  # PyTorch 1.11+
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_environment_fingerprint():
    pip_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode()
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return {
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cudnn_version": torch.backends.cudnn.version(),
        "pip_freeze": pip_freeze.splitlines(),
    }

def tiny_json(base, model, id, seed, skorch_params, backend, notes):

  os.makedirs(base, exist_ok=True)
  tiny = {
      "model_name": model,
      "subject_id": id,
      "seed": seed,
      "bandpass": {"l_freq": 4.0, "h_freq": 38.0},
      "unit_scale_to_uV": 1e6,
      "ems": {"factor_new": 1e-3, "init_block_size": 750},
      "trial_start_offset_seconds": -0.5,
      "windowing": "create_windows_from_events(session split: 0train/1test)",
      "zscore_applied": False,
      "skorch_params": skorch_params,
      "backend":backend,
      "notes": notes
  }
  return tiny

def safe_model_config(model_config: dict) -> dict:
    """Convert model_config into a JSON-serializable dict."""
    safe_cfg = {}
    for k, v in model_config.items():
        if k == "model_class":
            # store full module path + class name
            safe_cfg[k] = f"{v.__module__}.{v.__name__}" if hasattr(v, "__module__") else str(v)
        elif k == "training":
            safe_training = {}
            for tk, tv in v.items():
                if tk == "optimizer":
                    # also store optimizer class name
                    safe_training[tk] = f"{tv.__module__}.{tv.__name__}" if hasattr(tv, "__module__") else str(tv)
                else:
                    # assume JSON-friendly scalar
                    safe_training[tk] = tv
            safe_cfg[k] = safe_training
        else:
            safe_cfg[k] = v if isinstance(v, (int, float, str, bool, type(None))) else str(v)
    return safe_cfg



In [17]:
def train_single_run(model_name, subject_id, seed, dataset):

    # 0. RNG reproducibility ---------------------------------------------------

    set_all_seeds(seed)
    rng_state_np    = np.random.get_state()
    rng_state_torch = torch.get_rng_state()
    env_fp          = get_environment_fingerprint()

    print(f"\n=== Processing Subject {subject_id} for seed {seed} ===")

    # 1. data ------------------------------------------------------------------
    # Load data
    train_set, test_set, train_subset, val_subset = load_subject_data_cached(dataset, subject_id)

    # Build simple tensors to compute stats on train windows only
    X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T)
    train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
    train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

    # Empirical bounds for later clipping in attack
    train_min = X_train.min(axis=(0,2), keepdims=True)
    train_max = X_train.max(axis=(0,2), keepdims=True)

    # expose indices explicitly
    train_idx = train_subset.indices
    val_idx   = val_subset.indices
    test_idx  = np.arange(len(test_set))

    # 2. model & config --------------------------------------------------------

    # Get model config
    config = MODEL_CONFIGS[model_name]

    # Extract model params from dataset, initialise model and set hyper-parameters
    # classes needed for clf
    classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
    n_classes = len(classes)
    n_channels = train_subset[0][0].shape[0]
    n_times = train_subset[0][0].shape[1]

    model = config['model_class'](
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
    )

    # Special handling for EEGMamba
    if model_name == 'EEGMamba':
        model.enable_moe(False)  # Use standard classifier for baseline. Paper explicitly removes moe modules for
                                # single use mamba
        # Enable MoE head instead of standard classifier (For multi-use only)
        model.use_moe = False

    # 4. fit -------------------------------------------------------------------

    # Create new classifier with best parameters
    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        train_split=predefined_split(val_subset),  # Use all training data
        optimizer=config['training']['optimizer'],
        optimizer__lr=config['training']['lr'],
        optimizer__weight_decay=config['training']['weight_decay'],
        batch_size=config['training']['batch_size'],
        callbacks=["accuracy"],
        device=device,
        classes=classes,
        max_epochs=500,
    )

    # Train on full training set
    clf.fit(train_subset, y=None)

    # 5. test accuracy ---------------------------------------------------------

    # Evaluate the model after training
    y_test = test_set.get_metadata().target
    test_accuracy = clf.score(test_set, y = y_test)

    # 6. save everything -------------------------------------------------------

    _save_run(model_name, subject_id, seed,
              clf, test_set, rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean, train_std, train_min, train_max,
              config, env_fp)

    return test_accuracy

# ==============================================================================

# Generate final baseline table
def create_baseline_table(results):
    """Create a nice table of baselines"""
    rows = []

    for model_name in results.keys():
        for subject_id in subjects:
            scores = results[model_name][subject_id]
            valid_scores = [s for s in scores if not np.isnan(s)]

            if valid_scores:
                mean_acc = np.mean(valid_scores)
                std_acc = np.std(valid_scores)
                n_valid = len(valid_scores)
            else:
                mean_acc = std_acc = n_valid = np.nan

            rows.append({
                'Model': model_name,
                'Subject': subject_id,
                'Mean_Accuracy': mean_acc,
                'Std_Accuracy': std_acc,
                'N_Valid_Runs': n_valid,
                'Individual_Scores': scores
            })

    return pd.DataFrame(rows)


In [18]:
def _save_run(model_name, subject_id, seed, clf, test_set,
              rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean=None, train_std=None,
              train_min=None, train_max=None,
              model_config: dict = None,
              env_fingerprint: dict = None,
              device: str=device):
    """
    clf          : fitted skorch net (clf.module_ is torch.nn.Module)
    test_set     : braindecode Dataset (getitem -> (x, y))
    *_idx        : np.ndarray of ints
    train_mean/std/min/max : arrays shaped (C,) or (1,C,1) (we'll serialize as lists)
    model_config : dict of arch + training hyperparams
    env_fingerprint : dict from get_environment_fingerprint()
    """

    base = f"{SAVE_DIR}/{model_name}/{model_name}_S{subject_id}_seed{seed}"
    os.makedirs(base, exist_ok=True)

    # ---- 0) Metadata header ----
    meta = {
        "model_name": model_name,
        "subject_id": int(subject_id),
        "seed": int(seed),
    }

    if model_config is not None:
        meta["model_config"] = safe_model_config(model_config)
    if env_fingerprint is not None:
        meta["environment"] = env_fingerprint
    json.dump(meta, open(f"{base}/meta.json", "w"), indent=2)

    # ---- 1) Checkpoint (state_dict + optimizer) ----
    torch.save({
        "state_dict": clf.module_.state_dict(),
        "optimizer": getattr(clf, "optimizer_", None).state_dict() if hasattr(clf, "optimizer_") else None,
        "seed": seed
    }, f"{base}/checkpoint.pth")

    # ---- 2) Training curves/history ----
    hist = clf.history_
    # Adjust keys if needed:
    train_acc = [e.get("train_accuracy", e.get("train_acc")) for e in hist]
    val_acc   = [e.get("valid_accuracy", e.get("val_acc")) for e in hist]
    train_loss= [e.get("train_loss") for e in hist]
    val_loss  = [e.get("valid_loss", e.get("val_loss")) for e in hist]
    curves = {"train_acc": train_acc, "val_acc": val_acc,
              "train_loss": train_loss, "val_loss": val_loss}
    json.dump(curves, open(f"{base}/curves.json", "w"), indent=2)

    # ---- 3) Test logits (CLEAN) + loss vector ----
    # Build X_test, y_test from the Dataset (no shuffling!)
    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
    y_test = np.array(test_set.get_metadata().target)                  # (N,)
    clf.module_.eval().to(device)
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
        logits_t = clf.infer(X_test_t)   # shape (N, num_classes)
        logits = logits_t.detach().cpu().numpy()
    np.save(f"{base}/test_logits_clean.npy", logits)
    np.save(f"{base}/y_test.npy", y_test)

    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    y_test_t = torch.tensor(y_test, device=device, dtype=torch.long)
    logits_ten = torch.tensor(logits, device=device, dtype=torch.float32)
    loss_vec = loss_fn(logits_ten, y_test_t).cpu().numpy()
    np.save(f"{base}/test_loss_vector.npy", loss_vec)

    # ---- 4) RNG states ----
    pickle.dump({"numpy": rng_state_np, "torch": rng_state_torch}, open(f"{base}/rng_state.pkl", "wb"))

    # ---- 5) Splits ----
    splits = {"train_idx": train_idx.tolist(),
              "val_idx":   val_idx.tolist(),
              "test_idx":  test_idx.tolist()}
    json.dump(splits, open(f"{base}/splits.json", "w"), indent=2)

    # ---- 6) Preprocessing statistics (per-channel) ----
    prep = {"zscore_applied": False}
    if train_mean is not None:
        prep["train_mean"] = np.array(train_mean).reshape(-1).tolist()
    if train_std is not None:
        prep["train_std"]  = np.array(train_std).reshape(-1).tolist()
    if train_min is not None:
        prep["train_min"]  = np.array(train_min).reshape(-1).tolist()
    if train_max is not None:
        prep["train_max"]  = np.array(train_max).reshape(-1).tolist()
    json.dump(prep, open(f"{base}/preprocessing.json", "w"), indent=2)

    # ---- 7) Attack metadata (placeholder file to append later) ----
    # You will fill this AFTER you run attacks; we create an empty schema now for consistency.
    attack_meta = {
        "whitebox": {},
        "blackbox": {}
    }
    json.dump(attack_meta, open(f"{base}/attack_metadata.json", "w"), indent=2)

    # ---- 8) README for the run folder ----
    with open(f"{base}/README.txt", "w") as f:
        f.write(
            "Artifacts:\n"
            "- checkpoint.pth: model+optimizer state_dict\n"
            "- curves.json: train/val accuracy/loss per epoch\n"
            "- test_logits_clean.npy: logits on test set (clean)\n"
            "- test_loss_vector.npy: per-sample CE loss on test set (clean)\n"
            "- y_test.npy: test labels\n"
            "- rng_state.pkl: RNG snapshots (numpy/torch)\n"
            "- splits.json: train/val/test indices (no leakage)\n"
            "- preprocessing.json: channelwise stats\n"
            "- attack_metadata.json: to be populated after attacks\n"
            "- meta.json: model/subject/seed, environment fingerprint\n"
        )

    # ---------------------------
    # Tiny JSON: per-run manifest
    # ---------------------------
    # grab skorch hyperparams (safe dict)
    try:
        skorch_params = {}
        for k, v in clf.get_params().items():
            if isinstance(v, (str, bool)):
                skorch_params[k] = v
            elif isinstance(v, numbers.Number):
                skorch_params[k] = float(v) if isinstance(v, float) else int(v)
    except Exception:
        skorch_params = {}

    # attach environment + backend determinism fingerprint
    backend = {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if hasattr(torch.version, "cuda") else None,
        "cudnn_version": torch.backends.cudnn.version(),
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    }

    # small cache manifest (see function below)

    tiny = tiny_json(
        base, model_name, subject_id, seed, skorch_params, backend,
        notes="baseline training run"
    )

    with open(f"{base}/tiny.json", "w") as f:
        json.dump(tiny, f, indent=2)


In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------------------------------------------------------------------
# Baseline Run
# ----------------------------------------------------------------------------------------------------------------

seeds = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

datasets = {
    "BNCIv2": ("BNCI2014001", 9),

    # Ended up not useing DEAP
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Define your models and their hyperparameters
MODEL_CONFIGS = {
    'EEGNet': {
        'model_class': EEGNetv4,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'DeepConvNet': {
        'model_class': Deep4Net,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'CTNet': {
        'model_class': CTNet,
        'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    }
}

# Store all results: results[model_name][subject_id] = [acc1, acc2, acc3, acc4, acc5]
all_results = defaultdict(lambda: defaultdict(list))


# Main loop
"""
Uncomment Main loop to run and save baslines
"""

# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{'='*60}")
#     print(f"RUNNING BASELINE FOR {model_name.upper()}")
#     print(f"{'='*60}")

#     for subject_id in subjects:
#         print(f"\n--- Subject {subject_id} ---")

#         subject_scores = []
#         for seed in seeds:
#             print(f"  Seed {seed}: RUNNING")
#             try:
#                 accuracy = train_single_run(model_name, subject_id, seed, dataset)
#                 subject_scores.append(accuracy)
#                 print(f"  Seed {seed}: {accuracy:.4f}")
#             except Exception as e:
#                 print(f"  Seed {seed}: FAILED ({e})")
#                 subject_scores.append(np.nan)

#         # Store results for this (model, subject) pair
#         all_results[model_name][subject_id] = subject_scores

#         # Calculate stats for this subject
#         valid_scores = [s for s in subject_scores if not np.isnan(s)]
#         if valid_scores:
#             mean_acc = np.mean(valid_scores)
#             std_acc = np.std(valid_scores)
#             print(f"  Subject {subject_id} baseline: {mean_acc:.4f} ± {std_acc:.4f}")
#         else:
#             print(f"  Subject {subject_id}: ALL RUNS FAILED")

# # Create and display results
# baseline_df = create_baseline_table(all_results)
# print(f"\n{'='*80}")
# print("FINAL BASELINE RESULTS")
# print(f"{'='*80}")

# # Subject-wise baselines
# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{model_name}:")
#     model_data = baseline_df[baseline_df['Model'] == model_name]

#     subject_means = []
#     for _, row in model_data.iterrows():
#         if not np.isnan(row['Mean_Accuracy']):
#             print(f"  Subject {row['Subject']}: {row['Mean_Accuracy']:.4f} ± {row['Std_Accuracy']:.4f}")
#             subject_means.append(row['Mean_Accuracy'])
#         else:
#             print(f"  Subject {row['Subject']}: FAILED")

#     # Dataset-wide average
#     if subject_means:
#         dataset_mean = np.mean(subject_means)
#         dataset_std = np.std(subject_means)
#         print(f"  → Dataset average: {dataset_mean:.4f} ± {dataset_std:.4f}")
#     else:
#         print(f"  → Dataset average: FAILED")

# # Save results
# baseline_df.to_csv('baseline_results.csv', index=False)
# print(f"\nResults saved to baseline_results.csv")

'\nUncomment Main loop to run and save baslines\n'

# Adversarial attacks


## Initialize model and load checkpoints

In [78]:
# 0) tensors and stats
def load_adv_test_data(subject_id):

  train_set, test_set, train_subset, val_subset = load_subject_data_cached("BNCI2014_001", subject_id)

  X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T

  X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T
  y_test = np.array(test_set.get_metadata().target)                   # (N,)

  train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
  train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

  # Optionally, keep empirical bounds for later clipping in attack
  train_min = X_train.min(axis=(0,2), keepdims=True)
  train_max = X_train.max(axis=(0,2), keepdims=True)

  X = torch.tensor(X_test, dtype=torch.float32, device=device)  # (N,C,T)
  y = torch.tensor(y_test, dtype=torch.long, device=device)


  train_min_t = torch.tensor(train_min, dtype=torch.float32, device=device)  # (1,C,1)
  train_max_t = torch.tensor(train_max, dtype=torch.float32, device=device)
  train_std_np = np.array(train_std).reshape(-1)  # (C,)

  return X, y, train_min_t, train_max_t, train_std_np

X, y, train_min_t, train_max_t, train_std_np = load_adv_test_data(1)

print(X.shape)
print(y.shape)
print(train_min_t.shape)
print(train_max_t.shape)
print(train_std_np.shape)

n_channels = int(X.shape[1])
n_times    = int(X.shape[2])
n_classes  = int(y.max().item() + 1)  # assumes 0..K-1 labels
print(n_channels)
print(n_times)
print(n_classes)



torch.Size([288, 22, 1125])
torch.Size([288])
torch.Size([1, 22, 1])
torch.Size([1, 22, 1])
(22,)
22
1125
4


In [79]:

from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

# Print original CTNet keys
# print(model.state_dict().keys())
path = torch.load('/content/robust-eeg-models/results/CTNet/CTNet_S1_seed123/checkpoint.pth')
print(path.keys())
model.load_state_dict(path['state_dict'])   # <- no .eval() here
model.eval()


X, y, train_min_t, train_max_t, train_std_np = load_adv_test_data(1)

def _needs_4d_input(model, Xsample):
    try:
        with torch.no_grad():
            model(Xsample[:1])  # try 3D (N,C,T)
        return False
    except Exception:
        with torch.no_grad():
            model(Xsample[:1].unsqueeze(1))  # try 4D (N,1,C,T)
        return True
NEEDS_4D = _needs_4d_input(model, X)
print(NEEDS_4D)

/usr/local/lib/python3.12/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


dict_keys(['state_dict', 'optimizer', 'seed'])
False


## Initialise variables

In [ ]:
# eps_grid = [0.01, 0.02, 0.03, 0.05]
# batch_size = 128
# steps = 40
# alpha = eps/8
# restarts=5

## Set up, helper methods for adversarial attacks and eval mode

In [95]:
# ==== Adversarial Attack Configuration Runner (lean, library-first) ====
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchattacks as ta
from scipy.ndimage import gaussian_filter1d


def per_channel_clamp(x, vmin, vmax):
    return torch.max(torch.min(x, vmax), vmin)

def snr_db(x, x_adv):
    d = x_adv - x
    num = x.pow(2).sum((1,2)).sqrt()
    den = d.pow(2).sum((1,2)).sqrt().clamp_min(1e-12)
    return (20.0 * torch.log10(num/den)).detach().cpu().numpy()

def per_class_acc(y_true, y_pred, nclass=None):
    yt = y_true.detach().cpu().numpy(); yp = y_pred.detach().cpu().numpy()
    K = int(nclass) if nclass is not None else int(max(yp.max(), yt.max()) + 1)
    out=[]
    for c in range(K):
        idx = (yt == c)
        out.append(float((yp[idx] == c).mean()) if idx.any() else float("nan"))
    return out

def smooth_delta_gauss(delta_t: torch.Tensor, sigma_t: float) -> torch.Tensor:
    # delta_t: (N,C,T) on device; returns same shape/device

    #GPU Speed up
    if sigma_t <= 0:
      return delta_t
    device = delta_t.device
    T = delta_t.size(-1)

    radius = int(4 * sigma_t + 0.5)
    x = torch.arange(-radius, radius + 1, dtype=delta_t.dtype, device=device)
    kernel = torch.exp(-0.5 * (x / sigma_t).pow(2))
    kernel /= kernel.sum()                  # single 1-D Gaussian

    C = delta_t.size(1)                     # 22 channels
    kernel = kernel.expand(C, 1, -1)        # (C, 1, kernel_size)

    # (N, C, T) → convolve along T
    return F.conv1d(delta_t, kernel, padding=radius, groups=C)

    # # simple version
    # dn = delta_t.detach().cpu().numpy()
    # dn = gaussian_filter1d(dn, sigma=sigma_t, axis=-1, mode="nearest")
    # return torch.from_numpy(dn).to(delta_t.device)

# --------- budgets (μV) and mapping to normalized ε (scalar) ----------
muV_grid = [0.5, 1.0, 2.0]                     # per-channel μV budgets
def eps_from_muV(mu_v): return float(mu_v / median_std)   # scalar ε
def eps_uV_per_channel(eps): return (eps * train_std_np).tolist()

# --------- core runner: L_inf sweep (FGSM/PGD) ----------
def run_linf_sweep(atk_name, eps_list, steps=None, alpha_rule=lambda e: e/8, batch=128, seed=42):
    global model, X, y, train_min_t, train_max_t, train_std_np

    ctor = {"FGSM": ta.FGSM, "PGD": ta.PGD}[atk_name]
    rows=[]
    N = X.size(0)
    with torch.no_grad():
        num_classes = int(model(X[:1]).shape[-1])
        preds_clean = model(X).argmax(1).cpu()
        clean_acc = float((preds_clean == y.cpu()).float().mean().item())
        clean_acc_pc = per_class_acc(y.cpu(), preds_clean, num_classes)

    for mu_v in eps_list:
        eps = eps_from_muV(mu_v)
        kwargs = {}
        if atk_name == "PGD":
            kwargs["steps"] = steps if steps is not None else 40
            kwargs["alpha"] = float(alpha_rule(eps))
            kwargs["random_start"] = True

        atk = ctor(model, eps=float(eps), **kwargs)
        preds_adv=[]; snrs=[]; l2_all=[]; l2_succ=[]
        for i in range(0, N, batch):
            Xi = X[i:i+batch].detach().clone().requires_grad_(True)
            yi = y[i:i+batch]
            xa = atk(Xi, yi).detach()

            #Gausiann noise for easy physiologically plausible attack
            if lp_sigma_t is not None:         # pass lp_sigma_t to enable LP variant
              eps_here = eps                 # scalar ε already computed for this μV
              delta = xa - Xi
              delta = smooth_delta_gauss(delta, sigma_t=lp_sigma_t)
              delta = delta.clamp(-eps_here, eps_here)    # re-project to same L∞ budget
              xa = Xi + delta

            xa = per_channel_clamp(xa, train_min_t, train_max_t)
            with torch.no_grad():
                pa = model(xa).argmax(1)
            preds_adv.append(pa.cpu())
            d = (xa - Xi).flatten(1); l2v = d.norm(p=2, dim=1).detach().cpu().numpy()
            l2_all.extend(l2v)
            succ = (pa != yi).cpu().numpy()
            l2_succ.extend(l2v[succ]); snrs.extend(snr_db(Xi, xa))
        preds_adv = torch.cat(preds_adv)
        adv_acc = float((preds_adv == y.cpu()).float().mean().item())
        row = {
            "attack": f"{atk_name}_LP" if lp_sigma_t is not None else atk_name,
            "smooth": bool(lp_sigma_t is not None),
            "smooth_type": "gaussian" if lp_sigma_t is not None else None,
            "smooth_sigma_t": float(lp_sigma_t) if lp_sigma_t is not None else None,
            "norm": "Linf",
            "muV_budget": float(mu_v),
            "eps": float(eps),
            "eps_uV_per_channel": eps_uV_per_channel(eps),
            "steps": kwargs.get("steps"),
            "alpha": float(kwargs["alpha"]) if "alpha" in kwargs else None,
            "random_start": bool(kwargs.get("random_start", False)),
            "seed": seed,
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
            "ASR": 1.0 - adv_acc,
            "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
            "mean_L2_all": float(np.mean(l2_all)) if len(l2_all) else float("nan"),
            "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
            "clean_acc_per_class": clean_acc_pc,
            "adv_acc_per_class": per_class_acc(y.cpu(), preds_adv, num_classes),
            "restarts": 1, "targeted": False
        }
        rows.append(row)
    return rows

# --------- DeepFool (L2) ----------
def run_deepfool(batch=128, seed=42):
    atk = ta.DeepFool(model, steps=50)
    preds_adv=[]; snrs=[]; l2_all=[]; l2_succ=[]
    N=X.size(0)
    with torch.no_grad():
        preds_clean = model(X).argmax(1).cpu()
        clean_acc = float((preds_clean == y.cpu()).float().mean().item())
    for i in range(0, N, batch):
        Xi = X[i:i+batch].detach().clone().requires_grad_(True)
        yi = y[i:i+batch]
        xa = atk(Xi, yi).detach()
        xa = per_channel_clamp(xa, train_min_t, train_max_t)
        with torch.no_grad():
            pa = model(xa).argmax(1)
        preds_adv.append(pa.cpu())
        d = (xa - Xi).flatten(1); l2v = d.norm(p=2, dim=1).detach().cpu().numpy()
        l2_all.extend(l2v)
        succ = (pa != yi).cpu().numpy()
        l2_succ.extend(l2v[succ]); snrs.extend(snr_db(Xi, xa))
    preds_adv = torch.cat(preds_adv)
    adv_acc = float((preds_adv == y.cpu()).float().mean().item())
    return [{
        "attack": "DeepFool_L2",
        "norm": "L2",
        "muV_budget": None,
        "eps": None,
        "eps_uV_per_channel": None,
        "steps": 50, "alpha": None, "random_start": None,
        "seed": seed,
        "clean_acc": clean_acc, "adv_acc": adv_acc, "ASR": 1.0 - adv_acc,
        "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
        "mean_L2_all": float(np.mean(l2_all)) if len(l2_all) else float("nan"),
        "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
        "restarts": 1, "targeted": False
    }]

# --------- AutoAttack sanity (subset, mid μV) ----------##******** WILL BE DELETED***********
def run_autoattack_mid(batch=128, subset=512, mid_muV=1.0, seed=42):
    global model, X, y, train_min_t, train_max_t, train_std_np
    eps = eps_from_muV(mid_muV)
    idx = slice(0, min(subset, X.size(0)))
    Xs, ys = X[idx].detach().clone(), y[idx].detach().clone()

    # DEBUG: Check dimensions before AutoAttack
    print(f"DEBUG: Input to AutoAttack:")
    print(f"  Xs.shape: {Xs.shape}")
    print(f"  ys.shape: {ys.shape}")
    print(f"  Model expects: {X.shape}")

    # Test model directly first
    with torch.no_grad():
        test_output = model(Xs[:1])
        print(f"  Model output shape: {test_output.shape}")


    aa = AutoAttack(model, norm='Linf', eps=float(eps), version='custom', attacks_to_run=['apgd-ce', 'square'] )
    Xaa_adv = aa.run_standard_evaluation(Xs, ys, bs=batch)
    Xaa_adv = per_channel_clamp(Xaa_adv, train_min_t, train_max_t)
    with torch.no_grad():
        adv_acc = float((model(Xaa_adv).argmax(1) == ys).float().mean().item())
    # SNR / L2 on subset
    snrs = snr_db(Xs, Xaa_adv)
    l2v = (Xaa_adv - Xs).flatten(1).norm(p=2, dim=1).detach().cpu().numpy()
    return [{
        "attack": "AutoAttack_std",
        "norm": "Linf",
        "muV_budget": float(mid_muV),
        "eps": float(eps),
        "eps_uV_per_channel": eps_uV_per_channel(eps),
        "steps": None, "alpha": None, "random_start": None,
        "seed": seed,
        "subset_N": int(Xs.size(0)),
        "clean_acc": None,  # optional to fill
        "adv_acc": adv_acc, "ASR": 1.0 - adv_acc,
        "median_L2_success": float(np.median(l2v)),  # AA usually succeeds broadly; median over all
        "mean_L2_all": float(np.mean(l2v)),
        "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
        "restarts": None, "targeted": False
    }]

# Explainer  methods
def pick_subset(X_in, y_in, max_n=ATTR_MAX_N):
    if max_n and X_in.size(0) > max_n:
        return X_in[:max_n], y_in[:max_n]
    return X_in, y_in

def attr_LRP(X_in, y_in):
    Xi, yi = pick_subset(X_in, y_in)
    Xi = Xi.detach().clone().requires_grad_(True)
    if _HAS_ZENNIT:
        with torch.no_grad(): K = int(model(Xi[:1]).shape[-1])
        T = F.one_hot(yi, num_classes=K).float()
        comp = zcomp.EpsilonPlusFlat(rules={einops_torch.Rearrange: Pass()})
        with zattr.Attributor(model, comp) as attributor:
            _, R = attributor(Xi, T)
        return R
    # fallback: IG so pipeline doesn’t break
    return ig.attribute(Xi, target=yi, n_steps=N_STEPS_IG,
                        baselines=torch.zeros_like(Xi), internal_batch_size=IG_INT_BS)

def attr_IG(X_in, y_in):
    Xi, yi = pick_subset(X_in, y_in)
    Xi = Xi.detach().clone().requires_grad_(True)
    return ig.attribute(Xi, target=yi, n_steps=N_STEPS_IG,
                        baselines=torch.zeros_like(Xi), internal_batch_size=IG_INT_BS)


In [93]:
# ==== Per-subject, per-seed runner (uses previously defined attack functions) ====
import os, json, torch, pandas as pd
import captum.attr as CA

# --- REQUIRED: set your subject id and the 5 seeds you trained ---
SEEDS = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

# --- model builders and checkpoint paths ---
# lambdas to init the arch
MODEL_BUILDERS = {
    "EEGNet":      lambda: EEGNetv4(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "DeepConvNet": lambda: Deep4Net(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "CTNet":       lambda: CTNet(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
    "Mamba":       lambda: EEGMamba(n_chans=n_channels, n_outputs=n_classes, n_times=n_times),
}

# Map each model+seed to a ckpt path (state_dict). Adjust paths to your setup.
# Example pattern: f"checkpoints/{model}/{SUBJECT_ID}/seed{seed}.pt"
CKPTS = {
    "EEGNet":       {s: f"results/EEGNet/EEGNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"      for s in SEEDS},
    "DeepConvNet":  {s: f"results/DeepConvNet/DeepConvNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth" for s in SEEDS},
    "CTNet":        {s: f"results/CTNet/CTNet_S{SUBJECT_ID}_seed{s}/checkpoint.pth"       for s in SEEDS},
    "Mamba":        {s: f"results/EEGMamba/EEGMamba_S{SUBJECT_ID}_seed{s}/checkpoint.pth"       for s in SEEDS},
}

# Init Explainers
EXPLAINERS = {
    "LRP": attr_LRP,
    "IG":  attr_IG,
}

# ---- Explainers (LRP via Zennit + IG via Captum) ----
ATTR_MAX_N   = 128
N_STEPS_IG   = 16
IG_INT_BS    = 8

device = "cuda" if torch.cuda.is_available() else "cpu"


def run_for_model_seed(model_name: str, seed: int):
    global model  # reuse the attack functions' global `model`

    # ---- set random seeds ----
    set_all_seeds(seed)

    # ---- init model + load checkpoint ----
    m = MODEL_BUILDERS[model_name]().to(device)
    ckpt_path = CKPTS[model_name][seed]
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    # strip 'module.' if saved from DDP
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    m.load_state_dict(state_dict)
    m.eval()
    m = m.to(device)  # keeps model synced to CUDA
    model = m

    # Explanations calculated
    E_CLEAN = {}
    for name, fn in EXPLAINERS.items():
        E_CLEAN[name] = fn(X, y)   # capped to ATTR_MAX_N inside

    # ---- run the attacks exactly as configured earlier ----
    rows = []
    # standard
    rows += run_linf_sweep("FGSM", [0.5, 1.0, 2.0], steps=None,            lp_sigma_t=None)
    rows += run_linf_sweep("PGD",  [0.5, 1.0, 2.0], steps=40, alpha_rule=lambda e: e/8, lp_sigma_t=None)
    # low-pass variants
    rows += run_linf_sweep("FGSM", [0.5, 1.0, 2.0], steps=None,            lp_sigma_t=3.0)  # FGSM-LP
    rows += run_linf_sweep("PGD",  [0.5, 1.0, 2.0], steps=40, alpha_rule=lambda e: e/8, lp_sigma_t=3.0)  # PGD-LP

    rows += run_deepfool()
    # rows += run_autoattack_mid(mid_muV=1.0)

    # tag rows with identifiers
    for r in rows:
        r["subject_id"] = SUBJECT_ID
        r["model_name"] = model_name
        r["seed"] = seed
    return rows

def setup_subject(subject_id):
    global X, y, train_min_t, train_max_t, train_std_np, median_std
    X, y, train_min_t, train_max_t, train_std_np = load_adv_test_data(subject_id)
    median_std = float(np.median(train_std_np))   # used for μV -> ε mapping



SUBJECT_ID = 1
setup_subject(SUBJECT_ID)
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))

print(f"Data shape: {X.shape}, using n_times={n_times}")



# ---- main loop over models × seeds ----
all_rows = []
for model_name in MODEL_BUILDERS.keys():

    # NEED TO CHECK THIS TO MAKE SURE THE MODEL IS LOADING, KEEP GETTING A FALIEURE. ****************************
    try:
        print(f"Testing {model_name}...")
        model_test = MODEL_BUILDERS[model_name]()
        model_test.eval()
        with torch.no_grad():
          output = model_test(X[:1])
        print(f"{model_name} works fine, output: {output.shape}")
    except Exception as e:
        print(f"{model_name} FAILED: {e}")
    for seed in SEEDS:
        all_rows.extend(run_for_model_seed(model_name, seed))

# ---- save per-subject CSV (append-safe) ----
os.makedirs("results", exist_ok=True)
csv_path = f"results/adversarial_results_{SUBJECT_ID}.csv"
pd.DataFrame(all_rows).to_csv(csv_path, index=False)
print(f"Wrote {csv_path} with {len(all_rows)} rows.")

# ---- optional: also keep a master (append new subjects later) ----
master_path = "results/adversarial_results_MASTER.csv"
if os.path.isfile(master_path):
    pd.concat([pd.read_csv(master_path), pd.DataFrame(all_rows)], ignore_index=True).to_csv(master_path, index=False)
else:
    pd.DataFrame(all_rows).to_csv(master_path, index=False)
print(f"Updated {master_path}.")


/usr/local/lib/python3.12/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Data shape: torch.Size([288, 22, 1125]), using n_times=1125
Testing EEGNet..

EinopsError: Shape mismatch, 1125 != 1

## White‑box L∞ attacks (FGSM, BIM, PGD, MIM)

## L₂‑style attacks (DeepFool, CW)

## Masking sanity check (AutoAttack subset) + black‑box (Square, FAB)

# Save & pretty‑print

In [96]:
# Test each model individually with AutoAttack
for model_name in ["EEGNet", "DeepConvNet", "CTNet", "Mamba"]:  # Test braindecode models only
    try:
        print(f"\n=== Testing {model_name} with AutoAttack ===")

        # Load one model
        m = MODEL_BUILDERS[model_name]().to(device)
        ckpt_path = CKPTS[model_name][42]  # Test with first seed
        ckpt = torch.load(ckpt_path, map_location=device)
        state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
        if any(k.startswith("module.") for k in state_dict.keys()):
            state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

        m.load_state_dict(state_dict)
        m.eval()
        m = m.to(device)  # Ensure on correct device
        model = m

        # Test AutoAttack specifically
        rows = run_autoattack_mid(mid_muV=1.0, subset=64)  # Small subset for testing
        print(f"{model_name}: AutoAttack SUCCESS")

    except Exception as e:
        print(f"{model_name}: AutoAttack FAILED: {e}")


=== Testing EEGNet with AutoAttack ===
DEBUG: Input to AutoAttack:
  Xs.shape: torch.Size([64, 22, 1125])
  ys.shape: torch.Size([64])
  Model expects: torch.Size([288, 22, 1125])
  Model output shape: torch.Size([1, 4])
using custom version including apgd-ce, square.
initial accuracy: 64.06%
EEGNet: AutoAttack FAILED: Shape mismatch, 1125 != 1

=== Testing DeepConvNet with AutoAttack ===
DEBUG: Input to AutoAttack:
  Xs.shape: torch.Size([64, 22, 1125])
  ys.shape: torch.Size([64])
  Model expects: torch.Size([288, 22, 1125])
  Model output shape: torch.Size([1, 4])
using custom version including apgd-ce, square.
initial accuracy: 60.94%
DeepConvNet: AutoAttack FAILED: Shape mismatch, 1125 != 1

=== Testing CTNet with AutoAttack ===
DEBUG: Input to AutoAttack:
  Xs.shape: torch.Size([64, 22, 1125])
  ys.shape: torch.Size([64])
  Model expects: torch.Size([288, 22, 1125])
  Model output shape: torch.Size([1, 4])
using custom version including apgd-ce, square.
initial accuracy: 67.19%


## For TSNE and PCA illustrations

In [92]:
!rm -rf cache/

In [ ]:
with torch.no_grad():
    feats_clean = model.features(X)  # shape (N, D)
    feats_adv   = model.features(X_adv)
np.save(f"{base}/feats_clean_eps{eps}.npy", feats_clean.cpu().numpy())
np.save(f"{base}/feats_adv_eps{eps}.npy", feats_adv.cpu().numpy())


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))
tsne = TSNE(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))


# Single attack runner for PGD